# 📘 Track B — Notebook 1 (v2): Environment Setup & HuBERT Feature Extraction

**Project:** Synthetic Data Generation for Speech Emotion Recognition  
**Version:** v2 — Frame-level embeddings (128 × 768) instead of mean-pooled (1536,)

---

## Key Change from v1
Instead of collapsing the HuBERT output to a single vector via mean+std pooling,
we now keep the **full temporal sequence** resampled to exactly 128 frames.
This preserves speech dynamics needed to reconstruct realistic audio later.

- **v1 embedding shape:** `(1536,)` — lost all temporal info
- **v2 embedding shape:** `(128, 768)` — full frame sequence preserved

---

## Dataset Structure
```
my Dataset/
  1/  ← Actor (1–8)
    session1/  ← Session (session1–session5)
      anger/
        1.1.anger-01.wav  ← {actor}.{session}.{emotion}-{sentence}.wav
```

## Output
- `file_index.csv`
- `embeddings/actor*_sent*_*.npy` — shape `(5, 128, 768)` per group
- `embeddings/embedding_matrix.npy` — shape `(3200, 128, 768)`
- `embeddings/embedding_labels.csv`

---
## CELL 1 — Install Dependencies

In [ ]:
import sys
!{sys.executable} -m pip install transformers torchaudio librosa soundfile pandas numpy tqdm ipywidgets scikit-learn matplotlib seaborn scipy -q
print('✅ All packages installed.')

---
## CELL 2 — Environment & GPU Check

In [ ]:
import sys, os, platform, time
import torch
import torchaudio
import transformers
import librosa
import numpy as np
import pandas as pd
import matplotlib
import sklearn

print('=' * 55)
print('       ENVIRONMENT & GPU HEALTH CHECK')
print('=' * 55)
print(f'  Python        : {sys.version.split()[0]}')
print(f'  PyTorch       : {torch.__version__}')
print(f'  Torchaudio    : {torchaudio.__version__}')
print(f'  Transformers  : {transformers.__version__}')
print(f'  Librosa       : {librosa.__version__}')
print(f'  NumPy         : {np.__version__}')
print(f'  Scikit-learn  : {sklearn.__version__}')
print()

cuda_available = torch.cuda.is_available()
print(f'  CUDA Available : {cuda_available}')
if cuda_available:
    print(f'  GPU            : {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  VRAM           : {vram:.1f} GB')
    device = torch.device('cuda')
    print('  ✅ GPU ready.')
else:
    device = torch.device('cpu')
    print('  ⚠️  No GPU. CPU inference will be slow.')

print('=' * 55)
print(f'  Active device  : {device}')
print('=' * 55)

---
## CELL 3 — Configuration

In [ ]:
from pathlib import Path

# ============================================================
#  ✏️  EDIT THIS PATH to point to your 'my Dataset' folder
# ============================================================
DATA_ROOT = Path('./my_Dataset')

EMBEDDINGS_DIR = Path('./embeddings')
OUTPUTS_DIR    = Path('./outputs')
EMBEDDINGS_DIR.mkdir(exist_ok=True)
OUTPUTS_DIR.mkdir(exist_ok=True)

# HuBERT config
HUBERT_MODEL  = 'facebook/hubert-base-ls960'
EXTRACT_LAYER = 9
TARGET_SR     = 16000
MAX_DURATION  = 6.0

# Frame-level embedding config (KEY CHANGE FROM v1)
TARGET_FRAMES = 128   # Every embedding resampled to exactly 128 time frames
HUBERT_DIM    = 768   # Hidden size of hubert-base
# Final embedding shape per file: (TARGET_FRAMES, HUBERT_DIM) = (128, 768)

# Dataset labels
EMOTIONS    = ['anger', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'sarcastic', 'surprise']
N_ACTORS    = 8
N_SESSIONS  = 5
N_SENTENCES = 10

print('Configuration loaded:')
print(f'  Data root      : {DATA_ROOT}')
print(f'  HuBERT model   : {HUBERT_MODEL}')
print(f'  Extract layer  : {EXTRACT_LAYER}')
print(f'  Target frames  : {TARGET_FRAMES}')
print(f'  HuBERT dim     : {HUBERT_DIM}')
print(f'  Embedding shape: ({TARGET_FRAMES}, {HUBERT_DIM}) per file')

assert DATA_ROOT.exists(), f'❌ DATA_ROOT not found: {DATA_ROOT}'
print(f'\n✅ DATA_ROOT exists.')

---
## CELL 4 — Parse Dataset into File Index

In [ ]:
import re
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='muted')

def parse_filename(fname):
    pattern = r'^(\d+)\.(\d+)\.([a-zA-Z]+)-(\d+)\.wav$'
    match = re.match(pattern, fname)
    if not match:
        return None
    return {
        'actor'    : int(match.group(1)),
        'session'  : int(match.group(2)),
        'emotion'  : match.group(3).lower(),
        'sentence' : int(match.group(4)),
    }

def build_file_index(data_root):
    records, missing = [], []
    all_wavs = list(data_root.rglob('*.wav'))
    print(f'  Found {len(all_wavs)} .wav files')
    for fpath in tqdm(all_wavs, desc='Parsing filenames'):
        meta = parse_filename(fpath.name)
        if meta is None:
            missing.append(str(fpath))
            continue
        meta['path'] = str(fpath)
        records.append(meta)
    if missing:
        print(f'  ⚠️  {len(missing)} files did not match pattern:')
        for p in missing[:3]: print(f'     {p}')
    df = pd.DataFrame(records).sort_values(
        ['actor','session','emotion','sentence']).reset_index(drop=True)
    return df

# Check if index already exists to save time
index_path = Path('./file_index.csv')
if index_path.exists():
    df_index = pd.read_csv(index_path)
    print(f'✅ Loaded existing file_index.csv ({len(df_index)} records)')
else:
    print('Building file index...')
    df_index = build_file_index(DATA_ROOT)
    df_index.to_csv(index_path, index=False)
    print(f'✅ File index built: {len(df_index)} records')

df_index.head(5)

---
## CELL 5 — Dataset Distribution Check

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Dataset Distribution Check', fontsize=15, fontweight='bold')

emotion_counts = df_index['emotion'].value_counts().reindex(EMOTIONS)
axes[0].bar(emotion_counts.index, emotion_counts.values,
            color=sns.color_palette('muted', len(EMOTIONS)))
axes[0].set_title('Samples per Emotion')
axes[0].set_xlabel('Emotion'); axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)
for i, v in enumerate(emotion_counts.values):
    axes[0].text(i, v+1, str(v), ha='center', fontsize=9)

actor_counts = df_index['actor'].value_counts().sort_index()
axes[1].bar(actor_counts.index.astype(str), actor_counts.values, color='steelblue')
axes[1].set_title('Samples per Actor')
axes[1].set_xlabel('Actor ID'); axes[1].set_ylabel('Count')
for i, v in enumerate(actor_counts.values):
    axes[1].text(i, v+1, str(v), ha='center', fontsize=9)

pivot = df_index.groupby(['actor','emotion']).size().unstack(fill_value=0).reindex(columns=EMOTIONS)
sns.heatmap(pivot, ax=axes[2], annot=True, fmt='d', cmap='YlOrRd',
            linewidths=0.5, cbar_kws={'label': 'Sample Count'})
axes[2].set_title('Actor × Emotion Heatmap')
axes[2].set_xlabel('Emotion'); axes[2].set_ylabel('Actor')

plt.tight_layout()
plt.savefig('./outputs/dataset_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

group_counts  = df_index.groupby(['actor','sentence','emotion']).size()
incomplete    = group_counts[group_counts != N_SESSIONS]
print(f'Total files : {len(df_index)}')
if len(incomplete) == 0:
    print(f'✅ All groups have exactly {N_SESSIONS} samples.')
else:
    print(f'⚠️  {len(incomplete)} groups do not have {N_SESSIONS} samples.')
    print(incomplete)

---
## CELL 6 — Audio Sanity Check

In [ ]:
import IPython.display as ipd
import librosa.display

test_row  = df_index.iloc[0]
test_path = test_row['path']
print(f'Test file: {test_path}')

waveform, orig_sr = librosa.load(test_path, sr=None)
waveform_16k      = librosa.resample(waveform, orig_sr=orig_sr, target_sr=TARGET_SR)

print(f'  Original SR  : {orig_sr} Hz → {TARGET_SR} Hz')
print(f'  Duration     : {len(waveform_16k)/TARGET_SR:.2f}s')

fig, axes = plt.subplots(1, 2, figsize=(14, 3))
fig.suptitle(f'Audio Sanity Check — {test_row["emotion"]} (Actor {test_row["actor"]})')
librosa.display.waveshow(waveform_16k, sr=TARGET_SR, ax=axes[0])
axes[0].set_title('Waveform (16kHz)')
mel    = librosa.feature.melspectrogram(y=waveform_16k, sr=TARGET_SR, n_mels=80, fmax=8000)
mel_db = librosa.power_to_db(mel, ref=np.max)
img    = librosa.display.specshow(mel_db, sr=TARGET_SR, x_axis='time', y_axis='mel',
                                   fmax=8000, ax=axes[1], cmap='magma')
fig.colorbar(img, ax=axes[1], format='%+2.0f dB')
axes[1].set_title('Mel Spectrogram (80 bins)')
plt.tight_layout()
plt.savefig('./outputs/audio_sanity_check.png', dpi=150, bbox_inches='tight')
plt.show()
ipd.display(ipd.Audio(waveform_16k, rate=TARGET_SR))
print('✅ Audio loading test passed.')

---
## CELL 7 — Load HuBERT Model

**Skips download if model is already cached.**

In [ ]:
from transformers import HubertModel, Wav2Vec2FeatureExtractor

print(f'Loading HuBERT: {HUBERT_MODEL}')
print('(Uses cached version if already downloaded)')

t0 = time.time()
processor = Wav2Vec2FeatureExtractor.from_pretrained(HUBERT_MODEL)
hubert    = HubertModel.from_pretrained(HUBERT_MODEL, output_hidden_states=True)
hubert    = hubert.to(device)
hubert.eval()

elapsed     = time.time() - t0
total_params = sum(p.numel() for p in hubert.parameters()) / 1e6
print(f'\n✅ HuBERT loaded in {elapsed:.1f}s  |  {total_params:.1f}M params')
print(f'   Device: {next(hubert.parameters()).device}')
if torch.cuda.is_available():
    print(f'   VRAM : {torch.cuda.memory_allocated()/1e9:.2f} GB allocated')

---
## CELL 8 — Frame-Level Embedding Extractor

**KEY CHANGE FROM v1:**  
Returns `(TARGET_FRAMES, HUBERT_DIM)` = `(128, 768)` instead of `(1536,)`.  
Temporal structure is fully preserved via linear interpolation of the time axis.

In [ ]:
from scipy.interpolate import interp1d

def extract_embedding(waveform_np: np.ndarray, sr: int) -> np.ndarray:
    """
    Extract frame-level HuBERT embeddings from a raw waveform.

    v2 change: returns (TARGET_FRAMES, HUBERT_DIM) = (128, 768)
    instead of v1's mean+std pooled (1536,) vector.

    Steps:
    1. Resample to 16kHz
    2. Trim to MAX_DURATION
    3. Run HuBERT, extract layer EXTRACT_LAYER → (T, 768)
    4. Linearly interpolate time axis to exactly TARGET_FRAMES
    5. Return (TARGET_FRAMES, 768)
    """
    # Resample if needed
    if sr != TARGET_SR:
        waveform_np = librosa.resample(waveform_np, orig_sr=sr, target_sr=TARGET_SR)

    # Trim to max duration
    max_samples = int(MAX_DURATION * TARGET_SR)
    if len(waveform_np) > max_samples:
        waveform_np = waveform_np[:max_samples]

    # HuBERT feature extraction
    inputs = processor(waveform_np, sampling_rate=TARGET_SR,
                       return_tensors='pt', padding=True)
    input_values = inputs.input_values.to(device)

    with torch.no_grad():
        outputs = hubert(input_values, output_hidden_states=True)

    # Layer output: (1, T, 768) → (T, 768)
    layer_output = outputs.hidden_states[EXTRACT_LAYER]
    layer_output = layer_output.squeeze(0).cpu().numpy()  # (T, 768)

    # Resample time axis to exactly TARGET_FRAMES using linear interpolation
    T, D = layer_output.shape
    if T != TARGET_FRAMES:
        x_old        = np.linspace(0, 1, T)
        x_new        = np.linspace(0, 1, TARGET_FRAMES)
        interpolator = interp1d(x_old, layer_output, axis=0, kind='linear')
        layer_output = interpolator(x_new)  # (TARGET_FRAMES, 768)

    return layer_output.astype(np.float32)  # (128, 768)


# --- Single-file test ---
print('Testing frame-level extraction on one file...')
test_waveform, test_sr = librosa.load(test_path, sr=None)
test_embedding = extract_embedding(test_waveform, test_sr)

print(f'  Input waveform shape   : {test_waveform.shape}')
print(f'  Output embedding shape : {test_embedding.shape}  ← should be (128, 768)')
print(f'  Value range            : [{test_embedding.min():.4f}, {test_embedding.max():.4f}]')
print(f'  Temporal std (mean)    : {test_embedding.std(axis=0).mean():.4f}')

assert test_embedding.shape == (TARGET_FRAMES, HUBERT_DIM), \
    f'❌ Shape mismatch: got {test_embedding.shape}, expected ({TARGET_FRAMES}, {HUBERT_DIM})'
print('\n✅ Frame-level embedding test passed.')

if torch.cuda.is_available():
    print(f'   VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB')

---
## CELL 9 — Full Extraction Loop

**Skips extraction if embeddings already exist** to avoid rerunning on restart.

In [ ]:
from collections import defaultdict

matrix_path = EMBEDDINGS_DIR / 'embedding_matrix.npy'
labels_path = EMBEDDINGS_DIR / 'embedding_labels.csv'

# Check if embeddings already exist AND are v2 shape
need_rerun = True
if matrix_path.exists() and labels_path.exists():
    existing = np.load(matrix_path)
    if existing.ndim == 3 and existing.shape[1:] == (TARGET_FRAMES, HUBERT_DIM):
        print(f'✅ Embeddings already exist with correct v2 shape {existing.shape}. Skipping extraction.')
        emb_matrix = existing
        label_df   = pd.read_csv(labels_path)
        need_rerun = False
    else:
        print(f'⚠️  Existing embeddings have shape {existing.shape} (v1). Re-extracting with v2...')

if need_rerun:
    all_embeddings = []
    all_labels     = []
    group_data     = defaultdict(list)
    errors         = []
    t_start        = time.time()

    print(f'Extracting frame-level embeddings for {len(df_index)} files...')
    print(f'Output shape per file: ({TARGET_FRAMES}, {HUBERT_DIM})')

    for _, row in tqdm(df_index.iterrows(), total=len(df_index), desc='Extracting'):
        try:
            waveform, sr = librosa.load(row['path'], sr=None)
            emb          = extract_embedding(waveform, sr)  # (128, 768)

            all_embeddings.append(emb)
            all_labels.append({
                'actor'   : row['actor'],   'session' : row['session'],
                'emotion' : row['emotion'], 'sentence': row['sentence'],
            })
            group_key = (row['actor'], row['sentence'], row['emotion'])
            group_data[group_key].append(emb)

        except Exception as e:
            errors.append({'path': row['path'], 'error': str(e)})

    # Stack: (N, 128, 768)
    emb_matrix = np.stack(all_embeddings)
    label_df   = pd.DataFrame(all_labels)
    np.save(matrix_path, emb_matrix)
    label_df.to_csv(labels_path, index=False)

    # Save per-group files: each is (5, 128, 768)
    saved_groups = 0
    for (actor, sentence, emotion), emb_list in group_data.items():
        group_arr = np.stack(emb_list)
        fname     = EMBEDDINGS_DIR / f'actor{actor}_sent{sentence:02d}_{emotion}.npy'
        np.save(fname, group_arr)
        saved_groups += 1

    t_elapsed = time.time() - t_start
    print(f'\n✅ Extraction complete!')
    print(f'   Files processed : {len(all_embeddings)}')
    print(f'   Groups saved    : {saved_groups}')
    print(f'   Matrix shape    : {emb_matrix.shape}  ← (N, 128, 768)')
    print(f'   Time elapsed    : {t_elapsed/60:.1f} min')
    if errors:
        print(f'   ⚠️  {len(errors)} errors')
    else:
        print('   ✅ No errors.')
    if torch.cuda.is_available():
        print(f'   Peak VRAM: {torch.cuda.max_memory_allocated()/1e9:.2f} GB')

---
## CELL 10 — Visualize Temporal Structure of Embeddings

Confirms that temporal variation is preserved in the v2 embeddings.

In [ ]:
# Compare temporal std across emotions
emotion_temporal_std = {}
for emotion in EMOTIONS:
    mask = label_df['emotion'] == emotion
    embs = emb_matrix[mask]           # (N_emotion, 128, 768)
    # Mean std across frames per file, then average across files
    temporal_std = embs.std(axis=1).mean()  # scalar
    emotion_temporal_std[emotion] = temporal_std

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('v2 Embedding Quality — Temporal Structure Preserved', fontsize=13, fontweight='bold')

# Temporal std per emotion
emotions_sorted = list(emotion_temporal_std.keys())
stds            = [emotion_temporal_std[e] for e in emotions_sorted]
palette         = sns.color_palette('muted', len(EMOTIONS))
axes[0].bar(emotions_sorted, stds, color=palette)
axes[0].set_title('Mean Temporal Std per Emotion\n(higher = more temporal variation)')
axes[0].set_xlabel('Emotion'); axes[0].set_ylabel('Temporal Std')
axes[0].tick_params(axis='x', rotation=30)

# Visualize one sample embedding as a heatmap
sample_emb = emb_matrix[0]  # (128, 768)
# Show first 64 dims for clarity
im = axes[1].imshow(sample_emb[:, :64].T, aspect='auto', cmap='RdBu_r',
                     origin='lower', interpolation='nearest')
axes[1].set_title('Sample Embedding Heatmap\n(128 frames × 64 dims shown)')
axes[1].set_xlabel('Time Frame'); axes[1].set_ylabel('HuBERT Dim')
plt.colorbar(im, ax=axes[1])

plt.tight_layout()
plt.savefig('./outputs/embedding_temporal_structure.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTemporal std per emotion:')
for e, s in emotion_temporal_std.items():
    print(f'  {e:12s} : {s:.4f}')
print('\n✅ Notebook 1 complete. Proceed to Notebook 2.')
print(f'   Key output: embeddings/embedding_matrix.npy  shape={emb_matrix.shape}')